# Lecture 8 · Starter notebook — A fair comparison, in code
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture08/lecture08_1_fair_comparison.ipynb)

**Companion to [Chapter 2 of the video lecture](https://nlp-course-cuhksz.github.io/lecture08.html#p2-s18)** (slide 18) and to the research playbook on the
[final project guide](https://nlp-course-cuhksz.github.io/final-project.html#playbook). Every brief on the project page asks for the same thing: a baseline, **one change**,
and an honest comparison. The rule of thumb from the lecture is

> one change · matched budget · held-out split · independent checker · seeds · replayable traces.

This notebook turns those six words into about 150 lines of code you can copy into your own project. The "system" here is a
small text classifier (TF-IDF + a linear SVM on six newsgroups) because it trains in a second on a CPU; in your project
it might be an LLM prompt, an agent harness or a fine-tuned model. The code that does the comparison stays the same.

You will:
1. **freeze** a held-out test split and fingerprint it,
2. compare a baseline and one change under a **matched tuning budget**, then see what an unmatched budget does,
3. repeat over **seeds** and report the spread,
4. test whether the difference is real with a **paired bootstrap** and an exact **sign test**,
5. see why you must **never tune on the test set**,
6. check a **judge** against gold labels instead of trusting it, and
7. write **replayable traces** and a three-line (booktabs) LaTeX table for your report.

Runs on CPU in about two to five minutes (it downloads the 14 MB 20 Newsgroups archive once). No GPU and no API key needed.

In [ ]:
import numpy as np, json, hashlib, time
from scipy.stats import binomtest
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from functools import lru_cache
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import cohen_kappa_score

CATS = ["comp.graphics", "comp.windows.x", "rec.autos", "rec.motorcycles", "sci.med", "sci.space"]
raw = fetch_20newsgroups(subset="all", categories=CATS, remove=("headers", "footers", "quotes"), random_state=0)
texts, labels = raw.data, np.array(raw.target)
print(len(texts), "documents,", len(CATS), "classes:", ", ".join(CATS))

## 1 · Freeze the held-out split (week 1)
Decide the split **once**, with a fixed seed, before you run any experiment, and record a fingerprint of the test set in your
README. If the fingerprint ever changes, someone (maybe you) touched the test data. We keep three parts:
**train pool** (to fit models), **dev** (to tune and to make decisions) and **test** (looked at once, at the end).

In [ ]:
rng = np.random.default_rng(2026)
order = rng.permutation(len(texts))
test_idx, dev_idx, pool_idx = order[:1000], order[1000:1500], order[1500:]
fingerprint = hashlib.sha256("\x00".join(texts[i] for i in test_idx).encode("utf-8")).hexdigest()[:12]
print(f"train pool {len(pool_idx)} · dev {len(dev_idx)} · test {len(test_idx)} · test fingerprint {fingerprint}")

## 2 · A baseline and ONE change
Both conditions share everything (data, classifier, preprocessing, evaluation code) except one thing: the change adds
**word bigrams** to the features. Each run trains on a random sample of 1,500 documents from the pool; the seed decides the sample.
`run()` returns a 0/1 vector: was each evaluation item classified correctly? Every analysis below works on such vectors, so you can
plug in any system that produces them.

In [ ]:
CONDITIONS = {
    "baseline (unigrams)": dict(ngram_range=(1, 1)),
    "change (+ bigrams)":  dict(ngram_range=(1, 2)),
}
N_TRAIN = 1500
SPLITS = {"dev": dev_idx, "test": test_idx}

@lru_cache(maxsize=None)
def features(cond, seed, n_train=N_TRAIN):
    """Fit the vectorizer on the seed's sample of the pool (cached: same condition + seed -> same features)."""
    tr = np.random.default_rng(seed).choice(pool_idx, n_train, replace=False)
    vec = TfidfVectorizer(sublinear_tf=True, min_df=2, **CONDITIONS[cond])
    Xtr = vec.fit_transform([texts[i] for i in tr])
    Xs = {name: vec.transform([texts[i] for i in idx]) for name, idx in SPLITS.items()}
    return Xtr, labels[tr], Xs

def run(cond, C, seed, split, n_train=N_TRAIN):
    """Train a linear SVM with regularisation C; return per-item 0/1 correctness on a split, and the predictions."""
    Xtr, ytr, Xs = features(cond, seed, n_train)
    pred = LinearSVC(C=C).fit(Xtr, ytr).predict(Xs[split])
    return (pred == labels[SPLITS[split]]).astype(int), pred

t0 = time.time()
for cond in CONDITIONS:
    correct, _ = run(cond, C=1.0, seed=0, split="dev")
    print(f"{cond:22s} dev accuracy {correct.mean():.3f}")
print(f"({time.time() - t0:.1f}s)")

## 3 · Matched budget
Hyper-parameters and data are part of the budget. Here the budget is **how many values of the regularisation strength C you may try
on dev**, and **how many training documents** you get. A fair comparison gives both conditions the same tries (the same draws, even)
and the same data. Two unfair versions are very common in student reports: tune the new method carefully but run the baseline once
at a setting nobody checked, or give the new method more data. Compare the three gaps.

In [ ]:
def tune(cond, budget, seed, n_train=N_TRAIN):
    """Try `budget` values of C (log-uniform in [0.01, 3]) on dev; return the best C and its dev accuracy."""
    Cs = 10 ** np.random.default_rng(1000 + seed).uniform(-2, 0.5, budget)
    scores = [run(cond, C, seed, "dev", n_train)[0].mean() for C in Cs]
    return Cs[int(np.argmax(scores))], max(scores)

B, M = list(CONDITIONS)
fb, fm = tune(B, 4, 0)[1], tune(M, 4, 0)[1]
ub, um = run(B, 0.01, 0, "dev")[0].mean(), tune(M, 12, 0)[1]
db, dm = fb, tune(M, 4, 0, n_train=3000)[1]
print(f"matched (4 tries, 1,500 docs each):          baseline {fb:.3f}  change {fm:.3f}  gap {100*(fm-fb):+.1f} points")
print(f"unmatched tuning (1 unchecked try vs 12):    baseline {ub:.3f}  change {um:.3f}  gap {100*(um-ub):+.1f} points")
print(f"unmatched data (1,500 vs 3,000 documents):   baseline {db:.3f}  change {dm:.3f}  gap {100*(dm-db):+.1f} points")

Whatever the unmatched rows add on top of the matched gap comes from the extra tuning or the extra data, not from bigrams; a reader of
your report cannot tell the two apart unless you state the budget. In an agent project the same
applies to model calls: if your agent gets ten calls, the baseline should get ten tries too (for example best-of-10 with the same checker).

## 4 · Seeds: run more than once
One run is one sample. We now repeat the whole pipeline (sample training data, tune on dev with the same budget, evaluate once on test)
for five seeds, and log every run as a trace.

In [ ]:
SEEDS, BUDGET = [0, 1, 2, 3, 4], 4
traces = []
t0 = time.time()
for seed in SEEDS:
    for cond in CONDITIONS:
        C, dev_acc = tune(cond, BUDGET, seed)
        correct, pred = run(cond, C, seed, "test")            # the ONLY place the test set is used
        traces.append(dict(condition=cond, seed=seed, C=float(C), budget=BUDGET, n_train=N_TRAIN, dev_acc=float(dev_acc),
                           test_fingerprint=fingerprint, test_acc=float(correct.mean()), correct=correct.tolist(), pred=pred.tolist()))
print(f"{len(traces)} runs in {time.time() - t0:.0f}s")
for cond in CONDITIONS:
    accs = np.array([t["test_acc"] for t in traces if t["condition"] == cond])
    print(f"{cond:22s} test accuracy {accs.mean():.3f} ± {accs.std(ddof=1):.3f}  (per seed: {np.round(accs, 3).tolist()})")

## 5 · Is the difference real?
Both systems were scored on the **same** 1,000 test items, so compare them item by item (a *paired* test). Two simple, standard tools:

* **Paired bootstrap**: resample test items with replacement many times and look at the distribution of the accuracy difference.
  The 95% interval tells you how large the effect plausibly is.
* **Exact sign test (McNemar)**: only the items where the two systems disagree carry information. If the change were useless, each
  disagreement would go either way with probability ½.

For the bootstrap we average each item's difference over the five seeds; the sign test uses a single seed, because pooling seeds would
count the same test items five times and make the p-value look better than it is.

In [ ]:
def paired_bootstrap(a, b, n_boot=2000, seed=0):
    """a, b: 0/1 arrays (or stacked seeds x items). Returns mean diff (b - a) and a 95% interval, in accuracy points."""
    a, b = np.atleast_2d(a), np.atleast_2d(b)
    d = (b - a).astype(float).mean(0)                     # per-item difference, averaged over seeds
    r = np.random.default_rng(seed)
    boots = np.array([d[r.integers(0, len(d), len(d))].mean() for _ in range(n_boot)])
    return 100 * d.mean(), 100 * np.percentile(boots, 2.5), 100 * np.percentile(boots, 97.5)

A = np.array([t["correct"] for t in traces if t["condition"] == B])
Bm = np.array([t["correct"] for t in traces if t["condition"] == M])
diff, lo, hi = paired_bootstrap(A, Bm)
wins, losses = int(((Bm[0] == 1) & (A[0] == 0)).sum()), int(((Bm[0] == 0) & (A[0] == 1)).sum())
p = binomtest(wins, wins + losses, 0.5).pvalue
print(f"change - baseline: {diff:+.2f} points, 95% CI [{lo:+.2f}, {hi:+.2f}]  (1,000 items, averaged over {len(SEEDS)} seeds)")
print(f"seed 0: items the change fixed {wins}, items it broke {losses}, exact sign test p = {p:.3g}")
if lo > 0:
    print("verdict: the interval is above 0 -> the change helps (check that the effect is large enough to matter)")
elif hi < 0:
    print("verdict: the interval is below 0 -> the change hurts; a negative result worth explaining")
else:
    print("verdict: the interval includes 0 -> no evidence of a difference at this test-set size")

Write the result in your report exactly like this: *"x points (95% CI [lo, hi], 5 seeds, 1,000 test items)"*, next to the per-seed spread
from section 4. Then show one item the change fixed and one it broke: that is the start of your error analysis. And if the change does
not help, say so: a well-explained negative result is a perfectly good project (FutureSelf, [slide 73](https://nlp-course-cuhksz.github.io/lecture08.html#p6-s73)).

## 6 · Never tune on the test set
Suppose we try 30 reasonable configurations (either feature set, C between 0.1 and 1, ten different training samples) and pick the one with
the best **test** score. That score is now optimistic, because we selected it partly for being lucky on those items. We can see the size of
the effect by splitting the test set into a part we select on (200 items) and a fresh part we never looked at (800 items).
(For this demonstration only, we deliberately misuse the test set.)

In [ ]:
r = np.random.default_rng(7)
configs = [(B if r.integers(2) == 0 else M, float(10 ** r.uniform(-1, 0)), int(r.integers(10))) for _ in range(30)]
rows = []
for cond, C, seed in configs:
    c = run(cond, C, seed, "test")[0]
    rows.append((c[:200].mean(), c[200:].mean()))      # selection part (200 items) vs fresh part (800 items)
rows = np.array(rows)
best = int(np.argmax(rows[:, 0]))
rank = int((rows[:, 1] > rows[best, 1]).sum()) + 1
print(f"best of 30 configs on the selection part: {rows[best, 0]:.3f}")
print(f"the same config on the fresh part:        {rows[best, 1]:.3f}  (optimism {100*(rows[best,0]-rows[best,1]):+.1f} points; rank {rank} of 30 on fresh data)")
print(f"average config, selection vs fresh:       {rows[:, 0].mean():.3f} vs {rows[:, 1].mean():.3f}")

The winner's score on the part it was picked on is usually clearly higher than on fresh data, and it is often not even the best
configuration there, while an average configuration scores about the same on both parts: the gap is selection, not skill. With fewer test items or more configurations the optimism grows. This is
why the test split is frozen in week 1 and used once.

## 7 · An independent checker
Many projects need a **judge**: an LLM that grades answers, a reward model, a classifier that flags unsafe text. Never let a model grade
its own work, and never trust a judge you have not checked. Here we compare three ways of estimating our system's accuracy on 200
"audited" test items where we know the truth:

* **self-judge**: the system (a logistic-loss linear model) marks an answer correct when its own probability is above 0.5;
* **independent judge**: a different model (Naive Bayes) marks an answer correct when it agrees with it;
* **gold**: the real labels, standing in for a human audit of a sample.

In [ ]:
Xtr, ytr, Xs = features(B, 0)
audit = test_idx[:200]
Xa = Xs["test"][:200]
system = SGDClassifier(loss="log_loss", alpha=1e-4, random_state=0).fit(Xtr, ytr)
judge = MultinomialNB(alpha=0.1).fit(Xtr, ytr)

pred = system.predict(Xa)
gold_ok = (pred == labels[audit]).astype(int)
self_ok = (system.predict_proba(Xa).max(1) > 0.5).astype(int)
judge_ok = (judge.predict(Xa) == pred).astype(int)
for name, v in [("self-judge", self_ok), ("independent judge", judge_ok)]:
    print(f"{name:18s} says accuracy {v.mean():.3f} · agrees with gold on {np.mean(v == gold_ok):.1%} of items · Cohen's kappa {cohen_kappa_score(v, gold_ok):.2f}")
print(f"{'gold (audit)':18s} accuracy {gold_ok.mean():.3f}")

Compare each judge's estimate with the gold accuracy. In our test run the self-judge said 62.5% and the independent judge 92%, against
80% gold: both were wrong, in opposite directions, and neither number would have been evidence on its own. Whoever the judge is,
report its agreement with an audited sample (accuracy and Cohen's κ) next to every judge-based number. If agreement is low,
the judge's scores are not evidence. With an LLM judge the recipe is identical: label 50–200 items yourself, then measure agreement.

## 8 · Replayable traces and a three-line table
Save every run (configuration, seed, fingerprint, per-item predictions) to a JSON-lines file in your repository. Anyone, including a TA
checking the reproducibility bonus, can then rebuild your headline table from the log without retraining.

In [ ]:
with open("runs.jsonl", "w") as f:
    for t in traces:
        f.write(json.dumps(t) + "\n")

def table_from_log(path):
    runs = [json.loads(l) for l in open(path)]
    assert len({r["test_fingerprint"] for r in runs}) == 1, "runs were evaluated on different test sets!"
    out = {}
    for r in runs:
        out.setdefault(r["condition"], []).append(r["test_acc"])
    return {k: (100 * np.mean(v), 100 * np.std(v, ddof=1), len(v)) for k, v in out.items()}

tab = table_from_log("runs.jsonl")
best_cond = max(tab, key=lambda k: tab[k][0])
lines = [r"\begin{tabular}{lc}", r"\toprule", r"Method & Accuracy (\%) \\", r"\midrule"]
for k, (m, s, n) in tab.items():
    cell = f"{m:.1f} " + r"$\pm$" + f" {s:.1f}"
    if k == best_cond:
        cell = r"\textbf{" + cell + "}"
    lines.append(k + " & " + cell + r" \\")
lines += [r"\bottomrule", r"\end{tabular}"]
print("\n".join(lines))
print(f"% mean ± std over {tab[best_cond][2]} seeds; test fingerprint {fingerprint}; needs " + r"\usepackage{booktabs}")

That is the "consistent style" tip from the writing part of the lecture ([slide 94]({SITE}#p7-s94)): one rule at the top, one under
the header, one at the bottom, and the same number of decimals everywhere.

## 9 · Plug in your own system
To reuse this notebook, write one function that returns a 0/1 vector over your frozen evaluation items, for example:

```python
def run(condition, budget, seed, eval_items):
    # call your prompt / agent / fine-tuned model `budget` times per item at most,
    # score each item with an independent checker (tests, simulator, gold label, validated judge)
    return correct  # np.array of 0/1, one entry per item, same order every time
```

Then `paired_bootstrap`, `binomtest`, the trace file and `table_from_log` work unchanged.

## Exercises
1. **A different change.** Replace the bigram condition with character n-grams
   (`dict(analyzer="char_wb", ngram_range=(2, 5))`) and rerun sections 4–5. Is the effect bigger, and does the interval exclude zero?
2. **How many test items do you need?** Re-run section 5 on only the first 200 test items. How wide is the interval now? Roughly how
   many items would you need to detect a one-point difference? (Hint: the width shrinks like 1/√n.)
3. **Write the sentence.** Using your numbers from sections 3 and 5, write the two sentences of your report that state the result and
   the budget you gave each condition. Then read the [writing tips](https://nlp-course-cuhksz.github.io/writing-tips.html) and cut every
   word that adds nothing.